In [0]:

from pyspark.sql import functions as F

perf = spark.table(
    "workspace.default.freddie_perf_raw"
)

# Extract relevant monthly performance variables
p = (
    perf.select(
        F.col("_c0").alias("loan_id"),
        F.to_date(
            F.concat(F.col("_c1"), F.lit("01")),
            "yyyyMMdd"
        ).alias("report_month"),
        F.trim(F.col("_c3")).alias("status"),
        F.trim(F.col("_c8")).alias("zero_balance_code")
    )
    .withColumn(
        "delinq_months",
        F.when(
            F.col("status").rlike(r"^[0-9]{1,2}$"),
            F.col("status").cast("int")
        )
    )
    .withColumn(
        "status_group",
        F.when(F.col("delinq_months") == 0, "Current")
         .when(F.col("delinq_months") == 1, "30-59 DPD")
         .when(F.col("delinq_months") == 2, "60-89 DPD")
         .when(F.col("delinq_months") >= 3, "90+ DPD")
         .when(F.col("status") == "RA", "REO")
         .when(F.col("status") == "XX", "Unknown")
         .otherwise("Other / Missing")
    )
)

display(
    p.groupBy("status_group")
     .agg(
         F.count("*").alias("monthly_records"),
         F.countDistinct("loan_id").alias("unique_loans")
     )
     .orderBy(F.desc("monthly_records"))
)


status_group,monthly_records,unique_loans
Current,2771604,50000
90+ DPD,30357,2927
30-59 DPD,26350,7728
60-89 DPD,8372,3753
REO,370,40


In [0]:

# First observed 90+ DPD for each mortgage
first_event = (
    p.groupBy("loan_id")
    .agg(
        F.min("report_month").alias("first_observed"),
        F.max("report_month").alias("last_observed"),
        F.min(
            F.when(
                F.col("delinq_months") >= 3,
                F.col("report_month")
            )
        ).alias("first_90dpd")
    )
    .withColumn(
        "has_90dpd",
        F.col("first_90dpd").isNotNull().cast("int")
    )
)

display(
    first_event.agg(
        F.count("*").alias("total_loans"),
        F.sum("has_90dpd").alias("loans_with_90dpd"),
        F.round(
            100 * F.avg("has_90dpd"), 2
        ).alias("observed_event_pct")
    )
)

display(first_event.filter("has_90dpd = 1").limit(10))


total_loans,loans_with_90dpd,observed_event_pct
50000,2927,5.85


loan_id,first_observed,last_observed,first_90dpd,has_90dpd
F17Q10241439,2017-04-01,2021-02-01,2020-06-01,1
F17Q10243129,2017-04-01,2021-08-01,2020-06-01,1
F17Q10243298,2017-04-01,2026-06-01,2023-12-01,1
F17Q10252226,2017-04-01,2024-10-01,2020-04-01,1
F17Q10252456,2017-04-01,2020-11-01,2017-12-01,1
F17Q10260299,2017-05-01,2026-06-01,2019-11-01,1
F17Q10267283,2017-04-01,2024-11-01,2020-08-01,1
F17Q10290872,2019-03-01,2021-06-01,2020-07-01,1
F17Q20047115,2017-05-01,2020-09-01,2020-06-01,1
F17Q20055202,2017-05-01,2022-05-01,2020-01-01,1


In [0]:
# Zero Balance Code = column 9 (0-based index 8)
zero_balance = (
    perf
    .filter(
        F.col("_c8").isNotNull() &
        (F.trim(F.col("_c8")) != "")
    )
    .groupBy("_c8")
    .agg(
        F.count("*").alias("records"),
        F.countDistinct("_c0").alias("unique_loans")
    )
    .orderBy("_c8")
)

display(zero_balance)


_c8,records,unique_loans
01,37579,37579
02,35,35
03,8,8
09,34,34
15,26,26
16,112,112
96,105,105


In [0]:

orig = spark.table("workspace.default.freddie_orig_raw")

orig_dates = orig.select(
    F.col("_c19").alias("loan_id"),
    F.to_date(
        F.concat(F.col("_c1"), F.lit("01")),
        "yyyyMMdd"
    ).alias("first_payment_date")
)

first_reports = (
    perf.groupBy("_c0")
    .agg(F.min("_c1").alias("first_report_yyyymm"))
    .withColumnRenamed("_c0", "loan_id")
    .withColumn(
        "first_report_date",
        F.to_date(
            F.concat(
                F.col("first_report_yyyymm"),
                F.lit("01")
            ),
            "yyyyMMdd"
        )
    )
)

entry_audit = (
    orig_dates.join(first_reports, "loan_id")
    .withColumn(
        "entry_gap_months",
        F.months_between(
            "first_report_date",
            "first_payment_date"
        ).cast("int")
    )
)

display(
    entry_audit.agg(
        F.count("*").alias("total_loans"),
        F.min("entry_gap_months").alias("min_gap"),
        F.expr(
            "percentile_approx(entry_gap_months, 0.5)"
        ).alias("median_gap"),
        F.max("entry_gap_months").alias("max_gap"),
        F.sum(
            (F.col("entry_gap_months") >= 3).cast("int")
        ).alias("loans_with_3plus_month_gap")
    )
)


total_loans,min_gap,median_gap,max_gap,loans_with_3plus_month_gap
50000,-2,-1,34,600


In [0]:
monthly = (
    perf.select(
        F.col("_c0").alias("loan_id"),
        F.to_date(
            F.concat(F.col("_c1"), F.lit("01")),
            "yyyyMMdd"
        ).alias("report_month"),
        F.trim(F.col("_c3")).alias("status"),
        F.trim(F.col("_c8")).alias("zero_code"),
        F.to_date(
            F.concat(F.col("_c9"), F.lit("01")),
            "yyyyMMdd"
        ).alias("zero_date")
    )
    .withColumn(
        "dpd",
        F.when(
            F.col("status").rlike(r"^[0-9]{1,2}$"),
            F.col("status").cast("int")
        )
    )
)

# Summarize each loan's observed history
loan_audit = (
    monthly.groupBy("loan_id")
    .agg(
        F.min("report_month").alias("first_report"),
        F.max("report_month").alias("last_report"),
        F.min_by("status", "report_month")
            .alias("entry_status"),
        F.min(
            F.when(
                F.col("dpd") >= 3,
                F.col("report_month")
            )
        ).alias("first_90dpd"),
        F.min("zero_date").alias("termination_date"),
        F.max("zero_code").alias("termination_code"),
        F.sum(
            (
                F.col("zero_code").isNotNull() &
                F.col("zero_date").isNull()
            ).cast("int")
        ).alias("missing_termination_date")
    )
)

# Check event/termination timing and observation coverage
display(
    loan_audit.agg(
        F.count("*").alias("total_loans"),

        F.sum(
            (F.col("first_90dpd") ==
             F.col("first_report")).cast("int")
        ).alias("event_at_entry"),

        F.sum(
            (F.col("first_90dpd") <
             F.col("termination_date")).cast("int")
        ).alias("event_before_termination"),

        F.sum(
            (F.col("first_90dpd") ==
             F.col("termination_date")).cast("int")
        ).alias("event_same_month_as_termination"),

        F.sum(
            (F.col("first_90dpd") >
             F.col("termination_date")).cast("int")
        ).alias("event_after_termination"),

        F.sum(
            (F.col("termination_date").isNotNull() &
             F.col("first_90dpd").isNull()).cast("int")
        ).alias("terminated_without_90dpd"),

        F.sum("missing_termination_date")
            .alias("missing_termination_dates"),

        F.sum(
            (F.col("termination_date").isNull() &
             (F.col("last_report") <
              F.to_date(F.lit("2026-06-01")))).cast("int")
        ).alias("no_termination_before_cutoff")
    )
)

total_loans,event_at_entry,event_before_termination,event_same_month_as_termination,event_after_termination,terminated_without_90dpd,missing_termination_dates,no_termination_before_cutoff
50000,0,1643,10,0,36246,0,0


In [0]:



cutoff = F.to_date(F.lit("2026-06-01"))

survival = (
    loan_audit
    .withColumn(
        "same_month_tie",
        (
            F.col("first_90dpd").isNotNull() &
            (F.col("first_90dpd") == F.col("termination_date"))
        ).cast("int")
    )
    .withColumn(
        "event_type",
        F.when(
            F.col("first_90dpd").isNotNull() &
            (
                F.col("termination_date").isNull() |
                (F.col("first_90dpd") <= F.col("termination_date"))
            ),
            1
        )
        .when(F.col("termination_code") == "01", 2)
        .when(F.col("termination_date").isNotNull(), 3)
        .otherwise(0)
    )
    .withColumn(
        "exit_date",
        F.when(
            F.col("event_type") == 1,
            F.col("first_90dpd")
        )
        .when(
            F.col("event_type").isin(2, 3),
            F.col("termination_date")
        )
        .otherwise(cutoff)
    )
    .withColumn(
        "time_months",
        F.months_between(
            "exit_date", "first_report"
        ).cast("int")
    )
    .withColumn(
        "event",
        (F.col("event_type") == 1).cast("int")
    )
)

# Check final outcome counts
display(
    survival.groupBy("event_type")
    .agg(
        F.count("*").alias("n_loans"),
        F.round(
            F.avg("time_months"), 1
        ).alias("avg_followup_months")
    )
    .orderBy("event_type")
)


event_type,n_loans,avg_followup_months
0,10827,105.9
1,2927,37.0
2,36163,39.0
3,83,10.4


In [0]:

qc = survival.agg(
    F.count("*").alias("total_loans"),
    F.countDistinct("loan_id").alias("unique_loans"),
    F.sum("event").alias("events"),
    F.sum("same_month_tie").alias("ties"),
    F.min("time_months").alias("min_time"),
    F.max("time_months").alias("max_time"),
    F.sum(
        (F.col("time_months") < 0).cast("int")
    ).alias("negative_times"),
    F.sum(
        (
            (F.col("time_months") == 0) &
            (F.col("event_type") != 0)
        ).cast("int")
    ).alias("zero_time_exits"),
    F.sum(
        (
            (F.col("event") == 1) &
            (F.col("time_months") <= 36)
        ).cast("int")
    ).alias("events_within_36m"),
    F.sum(
        (F.col("exit_date") > F.to_date(F.lit("2026-06-01")))
        .cast("int")
    ).alias("exits_after_cutoff")
)

display(qc)


total_loans,unique_loans,events,ties,min_time,max_time,negative_times,zero_time_exits,events_within_36m,exits_after_cutoff
50000,50000,2927,10,0,113,0,32,1688,3


In [0]:



cutoff = F.to_date(F.lit("2026-06-01"))

exceptions = survival.filter(
    (F.col("time_months") == 0) |
    (F.col("exit_date") > cutoff)
)

display(
    exceptions.select(
        "loan_id",
        "first_report",
        "last_report",
        "first_90dpd",
        "termination_date",
        "termination_code",
        "event_type",
        "exit_date",
        "time_months"
    ).orderBy("exit_date")
)


loan_id,first_report,last_report,first_90dpd,termination_date,termination_code,event_type,exit_date,time_months
F17Q10049511,2017-02-01,2017-02-01,null,2017-02-01,01,2,2017-02-01,0
F17Q10141194,2017-03-01,2017-03-01,null,2017-03-01,01,2,2017-03-01,0
F17Q10207065,2017-04-01,2017-04-01,null,2017-04-01,01,2,2017-04-01,0
F17Q10249069,2017-04-01,2017-04-01,null,2017-04-01,01,2,2017-04-01,0
F17Q10227036,2017-04-01,2017-04-01,null,2017-04-01,01,2,2017-04-01,0
F17Q10160916,2017-04-01,2017-04-01,null,2017-04-01,01,2,2017-04-01,0
F17Q10246910,2017-04-01,2017-04-01,null,2017-04-01,01,2,2017-04-01,0
F17Q10248635,2017-04-01,2017-04-01,null,2017-04-01,01,2,2017-04-01,0
F17Q10220640,2017-04-01,2017-04-01,null,2017-04-01,01,2,2017-04-01,0
F17Q10227274,2017-05-01,2017-05-01,null,2017-05-01,01,2,2017-05-01,0


In [0]:



cutoff = F.to_date(F.lit("2026-06-01"))

display(
    survival.filter(F.col("exit_date") > cutoff)
    .select(
        "loan_id",
        "first_report",
        "last_report",
        "first_90dpd",
        "termination_date",
        "termination_code",
        "event_type",
        "exit_date"
    )
)


loan_id,first_report,last_report,first_90dpd,termination_date,termination_code,event_type,exit_date
F17Q20107477,2017-06-01,2026-06-01,null,2026-07-01,01,2,2026-07-01
F17Q20090098,2017-05-01,2026-06-01,null,2026-07-01,01,2,2026-07-01
F17Q40144607,2017-12-01,2026-06-01,null,2026-07-01,01,2,2026-07-01


In [0]:



cutoff = F.to_date(F.lit("2026-06-01"))

survival_final = (
    survival

    # Step 1: Flag future terminations
    .withColumn(
        "future_termination",
        (
            F.col("event_type").isin(2, 3) &
            (F.col("termination_date") > cutoff)
        ).cast("int")
    )

    # Step 2: Correct event type
    .withColumn(
        "event_type",
        F.when(
            F.col("future_termination") == 1, 0
        ).otherwise(F.col("event_type"))
    )

    # Step 3: Correct exit date
    .withColumn(
        "exit_date",
        F.when(
            F.col("future_termination") == 1, cutoff
        ).otherwise(F.col("exit_date"))
    )

    # Step 4: Recalculate follow-up time
    .withColumn(
        "time_months",
        F.months_between(
            "exit_date", "first_report"
        ).cast("int")
    )

    # Step 5: Ensure event indicator is consistent
    .withColumn(
        "event",
        (F.col("event_type") == 1).cast("int")
    )

    # Step 6: Exclude zero-time observations
    .filter(F.col("time_months") > 0)
)


In [0]:

qc = survival_final.agg(
    F.count("*").alias("total_loans"),
    F.countDistinct("loan_id").alias("unique_loans"),
    F.sum("event").alias("events"),
    F.min("time_months").alias("min_time"),
    F.max("time_months").alias("max_time"),
    F.sum(
        (F.col("exit_date") > cutoff).cast("int")
    ).alias("after_cutoff"),
    F.sum("future_termination").alias("future_terms_corrected")
)

display(qc)

display(
    survival_final.groupBy("event_type")
    .count()
    .orderBy("event_type")
)


total_loans,unique_loans,events,min_time,max_time,after_cutoff,future_terms_corrected
49968,49968,2927,1,113,0,3


event_type,count
0,10830
1,2927
2,36128
3,83


In [0]:

survival_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.freddie_survival"
    )

print("Survival outcome table saved successfully!")


Survival outcome table saved successfully!
